# Minimaltest: Qwen3 Embedding 0.6B

Dieses Notebook ist absichtlich minimal und testet nur, ob `Qwen/Qwen3-Embedding-0.6B-GGUF` lokal Embeddings erzeugen kann.

Evidenz aus der Modellseite:
- fuer llama.cpp soll das Modell mit `--pooling last` verwendet werden
- fuer Embeddings wird das GGUF-Modell `Q8_0` angeboten
- das Modell ist ein reines Text-Embedding-Modell

## Installation

```bash
pip install huggingface-hub llama-cpp-python
pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
```

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from tqdm.auto import tqdm

# Laut Qwen-Modellseite: llama.cpp Nutzung mit pooling last
LLAMA_POOLING_TYPE_LAST = 3

# Minimal sinnvolle Defaults fuer dein Setup
N_THREADS = 8
N_THREADS_BATCH = 8
N_CTX = 256
N_BATCH = 256

model_path = hf_hub_download(
    repo_id="Qwen/Qwen3-Embedding-0.6B-GGUF",
    filename="Qwen3-Embedding-0.6B-Q8_0.gguf",
)

embedding_llm = Llama(
    model_path=model_path,
    embedding=True,
    pooling_type=LLAMA_POOLING_TYPE_LAST,
    n_ctx=N_CTX,
    n_batch=N_BATCH,
    n_threads=N_THREADS,
    n_threads_batch=N_THREADS_BATCH,
    verbose=False,
)

In [ ]:
PARQUET_PATH = Path(r"C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260310_180458_ADS_Curation_Run\data\publications_disambiguated.parquet")
TEXT_COLUMN = "full_text"
N_DOCS = None  # None = alle Dokumente

df = pd.read_parquet(PARQUET_PATH)
docs_series = (
    df[TEXT_COLUMN]
    .dropna()
    .astype(str)
    .map(str.strip)
    .loc[lambda s: s != ""]
)

total_docs = len(docs_series)
if N_DOCS is None:
    docs = docs_series.tolist()
else:
    docs = docs_series.head(N_DOCS).tolist()

print(f"Insgesamt verwertbare Dokumente: {total_docs}")
print(f"Verarbeite {len(docs)} Dokumente aus {PARQUET_PATH.name}")

embeddings = []
for text in tqdm(docs, desc="Erzeuge Embeddings", unit="doc"):
    result = embedding_llm.create_embedding(text)
    embeddings.append(np.array(result["data"][0]["embedding"], dtype=np.float32))

embeddings = np.vstack(embeddings)
print("Embeddings erstellt.")
print("Shape:", embeddings.shape)
print("Erste 8 Werte des ersten Vektors:")
print(embeddings[0][:8])